## Ingest Fact Data from Bronze to Silver Layer

In [0]:
# Import Required Libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType,IntegerType,FloatType

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('order_source', 'order_items', 'order_source')
dbutils.widgets.text('return_source', 'order_returns', 'return_source')
dbutils.widgets.text('shipment_source', 'order_shipments', 'shipment_source')

## ORDER ITEMS

In [0]:
# set up the environment
catalog = dbutils.widgets.get("catalog")
order_source = dbutils.widgets.get("order_source")
print(catalog, order_source)


In [0]:
# Read Table
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{order_source}')
#display(df_bronze.limit(5))


- Perform Transformations and Cleaning

In [0]:
# print Schema
df_bronze.printSchema()

In [0]:
# Check null values in the columns
null_dt = df_bronze.filter(F.col('dt').isNull()).count()
print(null_dt)
null_order_ts = df_bronze.filter(F.col('order_ts').isNull()).count()
print(null_order_ts)
null_customer_id = df_bronze.filter(F.col('customer_id').isNull()).count()
print(null_customer_id)
null_order_id = df_bronze.filter(F.col('order_id').isNull()).count()
print(null_order_id)
null_item_seq = df_bronze.filter(F.col('customer_id').isNull()).count()
print(null_customer_id)
null_product_id = df_bronze.filter(F.col('product_id').isNull()).count()
print(null_product_id)
null_quantity = df_bronze.filter(F.col('quantity').isNull()).count()
print(null_quantity)
null_unit_price_curency = df_bronze.filter(F.col('unit_price_currency').isNull()).count()
print(null_unit_price_curency)
null_unit_price = df_bronze.filter(F.col('unit_price').isNull()).count()
print(null_unit_price)
null_discount_pct = df_bronze.filter(F.col('discount_pct').isNull()).count()
print(null_discount_pct)
null_tax_amount = df_bronze.filter(F.col('tax_amount').isNull()).count()
print(null_tax_amount)
null_channel = df_bronze.filter(F.col('channel').isNull()).count()
print(null_channel)
null_coupon_code = df_bronze.filter(F.col('coupon_code').isNull()).count()
print(null_coupon_code)


In [0]:
# change coupon code from null value to 'No coupon'
df_silver = df_bronze.fillna('No Coupon', subset = ["coupon_code"])
#display(df_silver.limit(10))

In [0]:
duplicates = df_bronze.groupby(df_silver.columns).count().filter('count > 1')
#display(duplicates)

In [0]:
# Transformation: Convert 'Two' → 2 and cast to Integer
df_silver = df_silver.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

In [0]:
# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df_silver = df_silver.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

In [0]:
# Transformation : Remove '%' from discount_pct and cast to double
df_silver = df_silver.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

#display(df_silver.limit(10))

In [0]:
# Transformation : coupon code processing (convert to lower)
df_silver = df_silver.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

In [0]:
# Transformation : channel processing 
df_silver = df_silver.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#display(df_silver.limit(5))

In [0]:
#Transformation : Add processed time 
df_silver = df_silver.withColumn(
    "processed_time", F.current_timestamp()
)

#display(df_silver.limit(5))

In [0]:
# Transformation : coupon code processing (convert to lower)
df_silver = df_silver.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)
#display(df_silver.limit(30))

In [0]:
# Save to Silver Table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{order_source}')

## ORDER RETURNS

In [0]:
# set up the environment
catalog = dbutils.widgets.get("catalog")
return_source = dbutils.widgets.get("return_source")
print(catalog, return_source)


In [0]:
# Read table 
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{return_source}')
#display(df_bronze.limit(5))

- Perform Transformations and Cleaning

In [0]:
# check null values in the table
null_return_ts = df_bronze.filter(F.col('return_ts').isNull()).count()
print(null_return_ts)
null_order_dt = df_bronze.filter(F.col('order_dt').isNull()).count()
print(null_order_dt)
null_order_id = df_bronze.filter(F.col('order_id').isNull()).count()
print(null_order_id)
null_reason = df_bronze.filter(F.col('reason').isNull()).count()
print(null_reason)


In [0]:
# check for duplicates in the table
dup_count = df_bronze.groupBy('order_id', 'order_dt', 'return_ts').count().filter(F.col('count') > 1).count()
print(dup_count)

In [0]:
# change order_dt to datetime format
df_silver = df_bronze.withColumn(
    'order_dt',
    F.to_timestamp(F.col('order_dt'), 'yyyy-MM-dd')
)
#display(df_silver.limit(5))

In [0]:
#Convert `reason` column to uppercase and trim whitespace.
df_silver = df_silver.withColumn(
    'reason',
    F.upper(F.trim(F.col('reason')))
)
#display(df_silver.limit(5))

In [0]:
# Add processed_time column
df_silver = df_silver.withColumn(
    'processed_time',
    F.current_timestamp()
)
#display(df_silver.limit(5))

In [0]:
#write to silver table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{return_source}')

## ORDER SHIPMENTS

In [0]:
catalog = dbutils.widgets.get('catalog')
shipment_source = dbutils.widgets.get('shipment_source')
print(catalog, shipment_source)

- Perform Transformations and Cleaning

In [0]:
# Read Table
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{shipment_source}')
#display(df_bronze.limit(5))

In [0]:
# check for null values in columns
null_order_dt = df_bronze.filter(F.col('order_dt').isNull()).count()
print(null_order_dt)
null_shipment_id = df_bronze.filter(F.col('shipment_id').isNull()).count()
print(null_shipment_id)
null_order_id = df_bronze.filter(F.col('order_id').isNull()).count()
print(null_order_id)
null_carrier = df_bronze.filter(F.col('carrier').isNull()).count()
print(null_carrier)


In [0]:
# check for duplicates
dup_count = df_bronze.groupby('order_dt', 'shipment_id', 'order_id', 'carrier').count().filter(F.col('count') > 1).count()
print(dup_count)


In [0]:
df_bronze = df_bronze.withColumn('order_dt', F.to_date(F.col('order_dt')))
df_bronze.printSchema()

In [0]:
# change order_dt to datetime format
df_silver = df_bronze.withColumn(
    'order_dt',
    F.to_timestamp(F.col('order_dt'), 'yyyy-MM-dd')
)
#display(df_silver.limit(5))

In [0]:
#Convert `carrier` column to uppercase and trim whitespace.
df_silver = df_silver.withColumn(
    'carrier',
    F.upper(F.trim(F.col('carrier')))
)
#display(df_silver.limit(5))

In [0]:
# Add processed_time column
df_silver = df_silver.withColumn(
    'processed_time',
    F.current_timestamp()
)
#display(df_silver.limit(5))

In [0]:
#write to silver table
df_silver.write\
    .format('delta')\
    .option('enable.ChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{shipment_source}')